# Mini-proyecto Python: Índice de resiliencia hídrica urbana y escenarios de adaptación

**Curso:** AM4315 – Seguridad Hídrica  
**Duración sugerida:** 2 a 3 horas  
**Herramientas:** `pandas`, `numpy`, `matplotlib`  
**Tipo de datos:** Datos sintéticos generados dentro del notebook

## Idea central

En esta práctica construiremos un ejercicio integrador para analizar la seguridad hídrica de un sistema urbano.  
La ciudad sintética será evaluada mediante oferta, demanda, déficit, estrés hídrico, confiabilidad, vulnerabilidad, recuperación e índice de resiliencia.

Luego compararemos escenarios de:

- cambio climático,
- crecimiento urbano,
- reducción de pérdidas,
- infraestructura natural,
- reúso de agua,
- combinación de presiones y medidas de adaptación.

El propósito no es predecir una ciudad real, sino practicar una lógica reproducible de diagnóstico y comparación de escenarios para apoyar decisiones de planificación hídrica.

## Objetivos de aprendizaje

Al finalizar el notebook, el estudiante podrá:

1. Crear un dataset mensual sintético para un sistema urbano de agua.
2. Calcular indicadores básicos de balance, déficit y estrés hídrico.
3. Visualizar series temporales de oferta, demanda, déficit y estrés.
4. Encapsular cálculos en funciones reutilizables.
5. Simular escenarios de presión y adaptación hídrica.
6. Construir un índice sintético de resiliencia hídrica urbana.
7. Comparar escenarios y formular una recomendación técnica.
8. Validar el notebook mediante pruebas simples con `assert`.

## Road map técnico de implementación

| Bloque | Tipo de celda | Contenido | Producto esperado |
|---|---|---|---|
| 0 | Markdown | Título, introducción y objetivos | Contexto de la práctica |
| 1 | Código | Importación de librerías y configuración general | Entorno listo |
| 2 | Markdown + código | Fase 1: dataset sintético | `df_base` con 36 meses |
| 3 | Código | Tests de Fase 1 | Validación estructural inicial |
| 4 | Markdown + código | Fase 2: diagnóstico base | Balance, déficit, estrés y resumen |
| 5 | Código | Tests de Fase 2 | Validación de indicadores |
| 6 | Markdown + código | Fase 3: visualización base | Tres gráficos diagnósticos |
| 7 | Código | Tests de Fase 3 | Validación de datos graficables |
| 8 | Markdown + código | Fase 4: funciones reutilizables | Funciones de análisis y gráficos |
| 9 | Código | Tests de Fase 4 | Validación funcional |
| 10 | Markdown + código | Fase 5: simulación de escenarios | Diccionario `escenarios` |
| 11 | Código | Tests de Fase 5 | Validación de escenarios |
| 12 | Markdown + código | Fase 6: índice de resiliencia | Funciones de resiliencia |
| 13 | Código | Tests de Fase 6 | Validación del índice |
| 14 | Markdown + código | Fase 7: tabla comparativa | `tabla_escenarios` |
| 15 | Código | Tests de Fase 7 | Validación de tabla |
| 16 | Markdown + código | Fase 8: gráficos comparativos | Gráficos entre escenarios |
| 17 | Código | Tests de Fase 8 | Validación de variables comparativas |
| 18 | Markdown | Fase 9: interpretación técnica | Preguntas y conclusión del estudiante |
| 19 | Código | Fase 10: validación final | Mensaje final de ejecución correcta |
| 20 | Markdown | Cierre de la práctica | Conclusiones generales |

## 0. Importación de librerías

Usaremos únicamente librerías básicas para mantener el ejercicio accesible:

- `numpy`: generación de datos sintéticos y operaciones numéricas.
- `pandas`: organización de datos en tablas.
- `matplotlib`: visualización.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configuración general para que los gráficos sean legibles.
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

# Semilla para que los resultados sean reproducibles.
np.random.seed(42)

print("Librerías importadas correctamente.")

# Fase 1 — Presentación del problema y creación del dataset sintético

## Objetivo

Crear una serie mensual de 36 meses para representar un sistema urbano de agua.

El dataset incluirá:

- fecha,
- mes,
- oferta de agua,
- demanda de agua,
- precipitación,
- población,
- pérdidas de red.

## Supuestos docentes

La ciudad sintética tiene una demanda creciente por aumento poblacional.  
La oferta depende parcialmente de la estacionalidad de la precipitación y se ve afectada por un episodio seco durante el segundo año.

La variable `demanda_m3s` representa la demanda efectiva del sistema, incluyendo pérdidas de red.  
Por eso, si las pérdidas aumentan, la demanda que debe cubrir el sistema también aumenta.

In [ ]:
# Generamos 36 fechas mensuales.
fechas = pd.date_range(start="2027-01-01", periods=36, freq="MS")
mes = fechas.month

# Patrón mensual de precipitación sintética en mm.
# Se usa una ciudad imaginaria con meses relativamente húmedos y meses secos.
patron_precipitacion = np.array([80, 95, 100, 70, 35, 20, 10, 12, 25, 40, 55, 75])
precipitacion_mm = np.tile(patron_precipitacion, 3).astype(float)

# Añadimos variabilidad aleatoria moderada.
precipitacion_mm = precipitacion_mm + np.random.normal(loc=0, scale=5, size=36)
precipitacion_mm = np.maximum(precipitacion_mm, 0)

# Episodio seco durante el segundo año, meses 16 a 21 del periodo.
precipitacion_mm[15:21] = precipitacion_mm[15:21] * 0.55

# Población inicial y crecimiento mensual aproximado.
poblacion_inicial = 4_800_000
crecimiento_mensual = 0.004
poblacion = poblacion_inicial * (1 + crecimiento_mensual) ** np.arange(36)
poblacion = poblacion.astype(int)

# Pérdidas de red en porcentaje.
# Se representa una red urbana con pérdidas cercanas a 28%, con ligera variación.
perdidas_red_pct = 28 + np.random.normal(loc=0, scale=1.2, size=36)
perdidas_red_pct = np.clip(perdidas_red_pct, 24, 32)

# Demanda neta por consumo poblacional.
# 150 litros/habitante/día = 0.150 m3/habitante/día.
consumo_per_capita_m3_dia = 0.150
demanda_neta_m3s = (poblacion * consumo_per_capita_m3_dia) / 86_400

# Demanda efectiva: consumo neto ajustado por pérdidas de red.
demanda_m3s = demanda_neta_m3s / (1 - perdidas_red_pct / 100)

# Oferta sintética.
# Depende de una base operativa y de un componente hidrológico asociado a precipitación.
oferta_base = 9.3
componente_hidrologico = 0.035 * precipitacion_mm
ruido_operativo = np.random.normal(loc=0, scale=0.25, size=36)
oferta_m3s = oferta_base + componente_hidrologico + ruido_operativo
oferta_m3s = np.maximum(oferta_m3s, 0.1)

# Construimos el DataFrame base.
df_base = pd.DataFrame({
    "fecha": fechas,
    "mes": mes,
    "oferta_m3s": oferta_m3s,
    "demanda_m3s": demanda_m3s,
    "precipitacion_mm": precipitacion_mm,
    "poblacion": poblacion,
    "perdidas_red_pct": perdidas_red_pct,
})

# Redondeamos solo para facilitar lectura.
df_base["oferta_m3s"] = df_base["oferta_m3s"].round(2)
df_base["demanda_m3s"] = df_base["demanda_m3s"].round(2)
df_base["precipitacion_mm"] = df_base["precipitacion_mm"].round(1)
df_base["perdidas_red_pct"] = df_base["perdidas_red_pct"].round(1)

df_base.head()

In [ ]:
# Tests de validación — Fase 1

columnas_requeridas_fase1 = [
    "fecha",
    "mes",
    "oferta_m3s",
    "demanda_m3s",
    "precipitacion_mm",
    "poblacion",
    "perdidas_red_pct",
]

assert len(df_base) == 36, "df_base debe tener 36 filas."
assert all(col in df_base.columns for col in columnas_requeridas_fase1), "Faltan columnas requeridas."
assert (df_base["oferta_m3s"] > 0).all(), "La oferta debe ser positiva."
assert (df_base["demanda_m3s"] > 0).all(), "La demanda debe ser positiva."
assert pd.api.types.is_datetime64_any_dtype(df_base["fecha"]), "La columna fecha debe ser de tipo fecha."

print("✅ Fase 1 validada: df_base fue creado correctamente.")

# Fase 2 — Diagnóstico hídrico del escenario base

## Objetivo

Calcular indicadores básicos para diagnosticar la situación del sistema urbano.

Usaremos cuatro columnas nuevas:

- `balance_m3s`: diferencia entre oferta y demanda.
- `deficit_m3s`: déficit cuando la demanda supera a la oferta.
- `indice_estres`: razón demanda/oferta.
- `hay_deficit`: variable booleana que identifica meses críticos.

## Interpretación conceptual

Una ciudad puede tener oferta promedio aparentemente suficiente y, aun así, enfrentar meses críticos.  
Por eso analizamos no solo promedios, sino también la frecuencia, magnitud y temporalidad del déficit.

In [ ]:
# Copiamos el DataFrame base para no perder la versión original.
df_diagnostico = df_base.copy()

# Balance hídrico simple.
df_diagnostico["balance_m3s"] = df_diagnostico["oferta_m3s"] - df_diagnostico["demanda_m3s"]

# El déficit solo existe cuando el balance es negativo.
df_diagnostico["deficit_m3s"] = np.where(
    df_diagnostico["balance_m3s"] < 0,
    -df_diagnostico["balance_m3s"],
    0
)

# Índice de estrés hídrico: demanda dividida entre oferta.
df_diagnostico["indice_estres"] = df_diagnostico["demanda_m3s"] / df_diagnostico["oferta_m3s"]

# Meses con déficit.
df_diagnostico["hay_deficit"] = df_diagnostico["deficit_m3s"] > 0

# Indicadores resumen.
meses_deficit = int(df_diagnostico["hay_deficit"].sum())
deficit_promedio = df_diagnostico["deficit_m3s"].mean()
deficit_maximo = df_diagnostico["deficit_m3s"].max()
mes_mas_critico = df_diagnostico.loc[df_diagnostico["deficit_m3s"].idxmax(), "fecha"]
estres_promedio = df_diagnostico["indice_estres"].mean()

print("Resumen del diagnóstico base")
print("----------------------------")
print(f"Meses con déficit: {meses_deficit}")
print(f"Déficit promedio: {deficit_promedio:.2f} m³/s")
print(f"Déficit máximo: {deficit_maximo:.2f} m³/s")
print(f"Mes más crítico: {mes_mas_critico.strftime('%Y-%m')}")
print(f"Índice de estrés promedio: {estres_promedio:.2f}")

df_diagnostico.head()

In [ ]:
# Tests de validación — Fase 2

assert (df_diagnostico["deficit_m3s"] >= 0).all(), "El déficit nunca debe ser negativo."
assert df_diagnostico["hay_deficit"].dtype == bool, "hay_deficit debe ser booleana."
assert (df_diagnostico["indice_estres"] > 0).all(), "El índice de estrés debe ser positivo."

sin_deficit = df_diagnostico["balance_m3s"] >= 0
con_deficit = df_diagnostico["balance_m3s"] < 0

assert (df_diagnostico.loc[sin_deficit, "deficit_m3s"] == 0).all(), "Si balance >= 0, déficit debe ser 0."
assert (df_diagnostico.loc[con_deficit, "deficit_m3s"] > 0).all(), "Si balance < 0, déficit debe ser mayor que 0."

print("✅ Fase 2 validada: indicadores de diagnóstico calculados correctamente.")

# Fase 3 — Visualización del diagnóstico

## Objetivo

Representar gráficamente el comportamiento del sistema base.

Se elaborarán tres gráficos:

1. Oferta vs demanda.
2. Déficit mensual.
3. Índice de estrés hídrico.

## Cómo interpretar los gráficos

- Cuando la demanda supera la oferta, aparece déficit.
- Un déficit persistente indica baja confiabilidad.
- Un índice de estrés mayor a 1 indica que la demanda supera la oferta disponible.

In [ ]:
# Gráfico 1: Oferta vs demanda
plt.figure()
plt.plot(df_diagnostico["fecha"], df_diagnostico["oferta_m3s"], marker="o", label="Oferta")
plt.plot(df_diagnostico["fecha"], df_diagnostico["demanda_m3s"], marker="o", label="Demanda")
plt.title("Oferta y demanda mensual de agua")
plt.xlabel("Fecha")
plt.ylabel("Caudal equivalente (m³/s)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Gráfico 2: Déficit mensual
plt.figure()
plt.bar(df_diagnostico["fecha"], df_diagnostico["deficit_m3s"], width=20)
plt.title("Déficit hídrico mensual")
plt.xlabel("Fecha")
plt.ylabel("Déficit (m³/s)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Gráfico 3: Índice de estrés hídrico
plt.figure()
plt.plot(df_diagnostico["fecha"], df_diagnostico["indice_estres"], marker="o", label="Índice de estrés")
plt.axhline(1, linestyle="--", label="Umbral demanda = oferta")
plt.title("Índice de estrés hídrico mensual")
plt.xlabel("Fecha")
plt.ylabel("Demanda / Oferta")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Tests de validación — Fase 3

columnas_graficos = ["fecha", "oferta_m3s", "demanda_m3s", "deficit_m3s", "indice_estres"]

assert all(col in df_diagnostico.columns for col in columnas_graficos), "Faltan columnas para graficar."
assert not df_diagnostico[columnas_graficos].isnull().any().any(), "No debe haber valores nulos en columnas graficadas."
assert len(df_diagnostico[columnas_graficos]) == 36, "Los datos graficables deben cubrir 36 meses."

print("✅ Fase 3 validada: datos listos para visualización.")

# Fase 4 — Construcción de funciones reutilizables

## Objetivo

Transformar el análisis en funciones para reutilizarlo en distintos escenarios.

Esto es clave en planificación hídrica porque permite comparar múltiples alternativas bajo una misma metodología.

In [ ]:
def calcular_indicadores_basicos(df):
    """
    Calcula balance, déficit, índice de estrés y condición de déficit.

    Parámetros
    ----------
    df : pandas.DataFrame
        Debe contener las columnas oferta_m3s y demanda_m3s.

    Retorna
    -------
    pandas.DataFrame
        Copia del DataFrame original con columnas nuevas.
    """
    df_resultado = df.copy(deep=True)

    df_resultado["balance_m3s"] = df_resultado["oferta_m3s"] - df_resultado["demanda_m3s"]
    df_resultado["deficit_m3s"] = np.where(
        df_resultado["balance_m3s"] < 0,
        -df_resultado["balance_m3s"],
        0
    )
    df_resultado["indice_estres"] = df_resultado["demanda_m3s"] / df_resultado["oferta_m3s"]
    df_resultado["hay_deficit"] = df_resultado["deficit_m3s"] > 0

    return df_resultado


def resumir_diagnostico(df):
    """
    Resume los indicadores principales de seguridad hídrica.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame con columnas de diagnóstico o con oferta y demanda.

    Retorna
    -------
    dict
        Diccionario con meses de déficit, déficit promedio, déficit máximo,
        estrés promedio y confiabilidad.
    """
    if not {"balance_m3s", "deficit_m3s", "indice_estres", "hay_deficit"}.issubset(df.columns):
        df = calcular_indicadores_basicos(df)

    resumen = {
        "meses_deficit": int(df["hay_deficit"].sum()),
        "deficit_promedio": float(df["deficit_m3s"].mean()),
        "deficit_maximo": float(df["deficit_m3s"].max()),
        "estres_promedio": float(df["indice_estres"].mean()),
        "confiabilidad": float((~df["hay_deficit"]).mean()),
    }

    return resumen


def graficar_oferta_demanda(df, titulo):
    """
    Grafica oferta y demanda mensual.
    """
    plt.figure()
    plt.plot(df["fecha"], df["oferta_m3s"], marker="o", label="Oferta")
    plt.plot(df["fecha"], df["demanda_m3s"], marker="o", label="Demanda")
    plt.title(titulo)
    plt.xlabel("Fecha")
    plt.ylabel("Caudal equivalente (m³/s)")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


def graficar_deficit(df, titulo):
    """
    Grafica déficit mensual.
    """
    plt.figure()
    plt.bar(df["fecha"], df["deficit_m3s"], width=20)
    plt.title(titulo)
    plt.xlabel("Fecha")
    plt.ylabel("Déficit (m³/s)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# Probamos las funciones con el escenario base.
df_base_funciones = calcular_indicadores_basicos(df_base)
resumen_base_funciones = resumir_diagnostico(df_base_funciones)

resumen_base_funciones

In [ ]:
# Tests de validación — Fase 4

df_original_prueba = df_base.copy(deep=True)
df_calculado_prueba = calcular_indicadores_basicos(df_original_prueba)

columnas_esperadas_funcion = ["balance_m3s", "deficit_m3s", "indice_estres", "hay_deficit"]

assert all(col not in df_original_prueba.columns for col in columnas_esperadas_funcion), (
    "calcular_indicadores_basicos no debe modificar el DataFrame original."
)

assert all(col in df_calculado_prueba.columns for col in columnas_esperadas_funcion), (
    "La función debe devolver las columnas de diagnóstico."
)

resumen_prueba = resumir_diagnostico(df_calculado_prueba)

assert isinstance(resumen_prueba, dict), "resumir_diagnostico debe devolver un diccionario."
assert 0 <= resumen_prueba["confiabilidad"] <= 1, "La confiabilidad debe estar entre 0 y 1."
assert isinstance(resumen_prueba["meses_deficit"], int), "meses_deficit debe ser entero."

print("✅ Fase 4 validada: funciones reutilizables operan correctamente.")

# Fase 5 — Simulación de escenarios

## Objetivo

Crear escenarios alternativos para comparar cómo cambia la seguridad y resiliencia hídrica urbana.

## Escenarios mínimos

1. Escenario base.
2. Cambio climático: reducción de oferta en 15%.
3. Crecimiento urbano: aumento de demanda en 20%.
4. Reducción de pérdidas: reducción de demanda efectiva en 10%.
5. Infraestructura natural: aumento de oferta en meses secos en 8%.
6. Reúso de agua: aumento constante de oferta en 1.5 m³/s.
7. Escenario combinado: cambio climático + crecimiento urbano + adaptación.

## Nota conceptual

Los escenarios no son predicciones. Son experimentos de planificación para explorar sensibilidad, comparar alternativas y discutir decisiones.

In [ ]:
def aplicar_escenario(
    df,
    nombre_escenario,
    reduccion_oferta=0,
    aumento_demanda=0,
    reduccion_demanda=0,
    aumento_oferta_constante=0,
    aumento_oferta_meses_secos=0
):
    """
    Aplica un escenario de presión o adaptación hídrica.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame base con oferta, demanda y precipitación.
    nombre_escenario : str
        Nombre del escenario.
    reduccion_oferta : float
        Proporción de reducción de oferta. Ejemplo: 0.15 = 15%.
    aumento_demanda : float
        Proporción de aumento de demanda. Ejemplo: 0.20 = 20%.
    reduccion_demanda : float
        Proporción de reducción de demanda efectiva. Ejemplo: 0.10 = 10%.
    aumento_oferta_constante : float
        Aumento constante de oferta en m3/s.
    aumento_oferta_meses_secos : float
        Aumento proporcional de oferta solo en meses secos.

    Retorna
    -------
    pandas.DataFrame
        DataFrame con escenario aplicado e indicadores recalculados.
    """
    df_escenario = df.copy(deep=True)

    # Modificación de oferta por presión climática.
    df_escenario["oferta_m3s"] = df_escenario["oferta_m3s"] * (1 - reduccion_oferta)

    # Modificación de demanda por crecimiento urbano.
    df_escenario["demanda_m3s"] = df_escenario["demanda_m3s"] * (1 + aumento_demanda)

    # Reducción de demanda efectiva por eficiencia o reducción de pérdidas.
    df_escenario["demanda_m3s"] = df_escenario["demanda_m3s"] * (1 - reduccion_demanda)

    # Aumento constante de oferta, por ejemplo por reúso de agua.
    df_escenario["oferta_m3s"] = df_escenario["oferta_m3s"] + aumento_oferta_constante

    # Infraestructura natural: aumento de oferta en meses secos.
    # Definimos meses secos como aquellos con precipitación menor o igual al percentil 40.
    umbral_seco = df_escenario["precipitacion_mm"].quantile(0.40)
    meses_secos = df_escenario["precipitacion_mm"] <= umbral_seco
    df_escenario.loc[meses_secos, "oferta_m3s"] = (
        df_escenario.loc[meses_secos, "oferta_m3s"] * (1 + aumento_oferta_meses_secos)
    )

    # Evitamos valores imposibles.
    df_escenario["oferta_m3s"] = df_escenario["oferta_m3s"].clip(lower=0.01)
    df_escenario["demanda_m3s"] = df_escenario["demanda_m3s"].clip(lower=0.01)

    # Recalculamos indicadores.
    df_escenario = calcular_indicadores_basicos(df_escenario)

    # Añadimos nombre de escenario.
    df_escenario["escenario"] = nombre_escenario

    return df_escenario


# Creamos los escenarios solicitados.
escenarios = {
    "Base": aplicar_escenario(df_base, "Base"),
    "Cambio climático (-15% oferta)": aplicar_escenario(
        df_base,
        "Cambio climático (-15% oferta)",
        reduccion_oferta=0.15
    ),
    "Crecimiento urbano (+20% demanda)": aplicar_escenario(
        df_base,
        "Crecimiento urbano (+20% demanda)",
        aumento_demanda=0.20
    ),
    "Reducción de pérdidas (-10% demanda)": aplicar_escenario(
        df_base,
        "Reducción de pérdidas (-10% demanda)",
        reduccion_demanda=0.10
    ),
    "Infraestructura natural (+8% oferta seca)": aplicar_escenario(
        df_base,
        "Infraestructura natural (+8% oferta seca)",
        aumento_oferta_meses_secos=0.08
    ),
    "Reúso de agua (+1.5 m³/s)": aplicar_escenario(
        df_base,
        "Reúso de agua (+1.5 m³/s)",
        aumento_oferta_constante=1.5
    ),
    "Combinado CC+urbano+adaptación": aplicar_escenario(
        df_base,
        "Combinado CC+urbano+adaptación",
        reduccion_oferta=0.15,
        aumento_demanda=0.20,
        reduccion_demanda=0.10,
        aumento_oferta_constante=1.5,
        aumento_oferta_meses_secos=0.08
    ),
}

# Revisamos un resumen rápido.
for nombre, df_esc in escenarios.items():
    resumen = resumir_diagnostico(df_esc)
    print(f"{nombre}: {resumen['meses_deficit']} meses con déficit, confiabilidad = {resumen['confiabilidad']:.2f}")

In [ ]:
# Tests de validación — Fase 5

assert all(len(df_esc) == 36 for df_esc in escenarios.values()), "Cada escenario debe conservar 36 filas."
assert escenarios["Cambio climático (-15% oferta)"]["oferta_m3s"].mean() < escenarios["Base"]["oferta_m3s"].mean(), (
    "Cambio climático debe reducir la oferta promedio."
)
assert escenarios["Crecimiento urbano (+20% demanda)"]["demanda_m3s"].mean() > escenarios["Base"]["demanda_m3s"].mean(), (
    "Crecimiento urbano debe aumentar la demanda promedio."
)
assert escenarios["Reducción de pérdidas (-10% demanda)"]["demanda_m3s"].mean() < escenarios["Base"]["demanda_m3s"].mean(), (
    "Reducción de pérdidas debe reducir la demanda promedio."
)
assert escenarios["Reúso de agua (+1.5 m³/s)"]["oferta_m3s"].mean() > escenarios["Base"]["oferta_m3s"].mean(), (
    "Reúso de agua debe aumentar la oferta promedio."
)
assert all("escenario" in df_esc.columns for df_esc in escenarios.values()), "Todos los escenarios deben tener columna escenario."

print("✅ Fase 5 validada: escenarios generados correctamente.")

# Fase 6 — Cálculo del índice de resiliencia hídrica urbana

## Objetivo

Construir un índice sintético simple para comparar escenarios.

## Indicadores

### 1. Confiabilidad

Proporción de meses sin déficit.

\[
confiabilidad = \frac{meses\ sin\ déficit}{meses\ totales}
\]

### 2. Vulnerabilidad normalizada

Magnitud promedio del déficit en los meses críticos, normalizada por la demanda promedio de esos meses.

\[
vulnerabilidad = \frac{déficit\ promedio\ en\ meses\ críticos}{demanda\ promedio\ en\ meses\ críticos}
\]

Si no hay déficit, la vulnerabilidad es 0.

### 3. Recuperación

Proporción de veces en que el sistema vuelve a un mes sin déficit después de un mes con déficit.

\[
recuperación = \frac{transiciones\ de\ déficit\ a\ no\ déficit}{meses\ con\ déficit\ que\ tienen\ mes\ siguiente}
\]

Si no hay déficit, la recuperación se considera 1.

### 4. Índice de resiliencia

\[
IRHU = 0.4 \times confiabilidad + 0.3 \times recuperación + 0.3 \times (1 - vulnerabilidad)
\]

El índice se acota entre 0 y 1.

In [ ]:
def calcular_confiabilidad(df):
    """
    Calcula la proporción de meses sin déficit.
    """
    if "hay_deficit" not in df.columns:
        df = calcular_indicadores_basicos(df)

    confiabilidad = (~df["hay_deficit"]).mean()
    return float(np.clip(confiabilidad, 0, 1))


def calcular_vulnerabilidad(df):
    """
    Calcula la vulnerabilidad normalizada por demanda en meses críticos.

    Retorna un valor entre 0 y 1.
    """
    if not {"deficit_m3s", "hay_deficit"}.issubset(df.columns):
        df = calcular_indicadores_basicos(df)

    df_critico = df[df["hay_deficit"]].copy()

    if len(df_critico) == 0:
        return 0.0

    deficit_promedio_critico = df_critico["deficit_m3s"].mean()
    demanda_promedio_critica = df_critico["demanda_m3s"].mean()

    vulnerabilidad = deficit_promedio_critico / demanda_promedio_critica
    return float(np.clip(vulnerabilidad, 0, 1))


def calcular_recuperacion(df):
    """
    Calcula la capacidad de recuperación después de meses con déficit.

    Retorna un valor entre 0 y 1.
    """
    if "hay_deficit" not in df.columns:
        df = calcular_indicadores_basicos(df)

    deficit = df["hay_deficit"].to_numpy()

    # Si no hay déficit, asumimos recuperación máxima.
    if deficit.sum() == 0:
        return 1.0

    # Solo cuentan meses con déficit que tienen un mes siguiente.
    deficit_con_mes_siguiente = deficit[:-1]
    total_oportunidades = deficit_con_mes_siguiente.sum()

    if total_oportunidades == 0:
        return 1.0

    recuperaciones = (deficit[:-1] == True) & (deficit[1:] == False)
    recuperacion = recuperaciones.sum() / total_oportunidades

    return float(np.clip(recuperacion, 0, 1))


def calcular_indice_resiliencia(df):
    """
    Calcula el índice sintético de resiliencia hídrica urbana.
    """
    confiabilidad = calcular_confiabilidad(df)
    vulnerabilidad = calcular_vulnerabilidad(df)
    recuperacion = calcular_recuperacion(df)

    indice = (
        0.4 * confiabilidad
        + 0.3 * recuperacion
        + 0.3 * (1 - vulnerabilidad)
    )

    return float(np.clip(indice, 0, 1))


# Ejemplo con el escenario base.
confiabilidad_base = calcular_confiabilidad(escenarios["Base"])
vulnerabilidad_base = calcular_vulnerabilidad(escenarios["Base"])
recuperacion_base = calcular_recuperacion(escenarios["Base"])
indice_base = calcular_indice_resiliencia(escenarios["Base"])

print(f"Confiabilidad base: {confiabilidad_base:.2f}")
print(f"Vulnerabilidad base: {vulnerabilidad_base:.2f}")
print(f"Recuperación base: {recuperacion_base:.2f}")
print(f"Índice de resiliencia base: {indice_base:.2f}")

In [ ]:
# Tests de validación — Fase 6

for nombre, df_esc in escenarios.items():
    c = calcular_confiabilidad(df_esc)
    v = calcular_vulnerabilidad(df_esc)
    r = calcular_recuperacion(df_esc)
    i = calcular_indice_resiliencia(df_esc)

    assert 0 <= c <= 1, f"Confiabilidad fuera de rango en {nombre}."
    assert 0 <= v <= 1, f"Vulnerabilidad fuera de rango en {nombre}."
    assert 0 <= r <= 1, f"Recuperación fuera de rango en {nombre}."
    assert 0 <= i <= 1, f"Índice de resiliencia fuera de rango en {nombre}."

# Prueba conceptual: un sistema sin déficit debe ser más resiliente que uno con déficit severo.
df_sin_deficit = df_base.copy()
df_sin_deficit["oferta_m3s"] = df_sin_deficit["demanda_m3s"] * 2
df_sin_deficit = calcular_indicadores_basicos(df_sin_deficit)

df_deficit_severo = df_base.copy()
df_deficit_severo["oferta_m3s"] = df_deficit_severo["demanda_m3s"] * 0.25
df_deficit_severo = calcular_indicadores_basicos(df_deficit_severo)

assert calcular_indice_resiliencia(df_sin_deficit) > calcular_indice_resiliencia(df_deficit_severo), (
    "Un escenario sin déficit debe tener mayor resiliencia que uno con déficit severo."
)

print("✅ Fase 6 validada: funciones de resiliencia calculan valores consistentes.")

# Fase 7 — Comparación de escenarios

## Objetivo

Crear una tabla comparativa con los principales indicadores de cada escenario.

La tabla permitirá responder:

- ¿Qué escenario tiene mayor resiliencia?
- ¿Qué escenario tiene mayor déficit?
- ¿Qué medidas de adaptación reducen mejor el riesgo?
- ¿Qué medidas resultan insuficientes por sí solas?

La tabla se ordenará desde el escenario más resiliente hasta el menos resiliente.

In [ ]:
filas_resumen = []

for nombre, df_esc in escenarios.items():
    resumen = resumir_diagnostico(df_esc)

    fila = {
        "escenario": nombre,
        "meses_deficit": resumen["meses_deficit"],
        "deficit_promedio": resumen["deficit_promedio"],
        "deficit_maximo": resumen["deficit_maximo"],
        "estres_promedio": resumen["estres_promedio"],
        "confiabilidad": calcular_confiabilidad(df_esc),
        "vulnerabilidad": calcular_vulnerabilidad(df_esc),
        "recuperacion": calcular_recuperacion(df_esc),
        "indice_resiliencia": calcular_indice_resiliencia(df_esc),
    }

    filas_resumen.append(fila)

tabla_escenarios = pd.DataFrame(filas_resumen)

# Ordenamos de mayor a menor resiliencia.
tabla_escenarios = tabla_escenarios.sort_values(
    by="indice_resiliencia",
    ascending=False
).reset_index(drop=True)

# Redondeamos para lectura.
tabla_escenarios_redondeada = tabla_escenarios.copy()
columnas_redondear = [
    "deficit_promedio",
    "deficit_maximo",
    "estres_promedio",
    "confiabilidad",
    "vulnerabilidad",
    "recuperacion",
    "indice_resiliencia",
]
tabla_escenarios_redondeada[columnas_redondear] = tabla_escenarios_redondeada[columnas_redondear].round(3)

tabla_escenarios_redondeada

In [ ]:
# Tests de validación — Fase 7

columnas_requeridas_tabla = [
    "escenario",
    "meses_deficit",
    "deficit_promedio",
    "deficit_maximo",
    "estres_promedio",
    "confiabilidad",
    "vulnerabilidad",
    "recuperacion",
    "indice_resiliencia",
]

assert set(tabla_escenarios["escenario"]) == set(escenarios.keys()), "La tabla debe contener todos los escenarios."
assert all(col in tabla_escenarios.columns for col in columnas_requeridas_tabla), "Faltan columnas en tabla_escenarios."
assert tabla_escenarios["indice_resiliencia"].between(0, 1).all(), "El índice debe estar entre 0 y 1."
assert tabla_escenarios["indice_resiliencia"].is_monotonic_decreasing, "La tabla debe estar ordenada de mayor a menor resiliencia."
assert not tabla_escenarios.isnull().any().any(), "La tabla no debe contener valores nulos."

print("✅ Fase 7 validada: tabla comparativa construida correctamente.")

# Fase 8 — Visualización comparativa

## Objetivo

Graficar los resultados comparativos entre escenarios.

Se elaborarán cuatro visualizaciones:

1. Índice de resiliencia por escenario.
2. Meses con déficit por escenario.
3. Déficit máximo por escenario.
4. Comparación de oferta y demanda para escenarios seleccionados.

In [ ]:
# Gráfico 1: Índice de resiliencia por escenario
plt.figure(figsize=(11, 4))
plt.bar(tabla_escenarios["escenario"], tabla_escenarios["indice_resiliencia"])
plt.title("Índice de resiliencia hídrica urbana por escenario")
plt.xlabel("Escenario")
plt.ylabel("Índice de resiliencia")
plt.ylim(0, 1)
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

# Gráfico 2: Meses con déficit por escenario
plt.figure(figsize=(11, 4))
plt.bar(tabla_escenarios["escenario"], tabla_escenarios["meses_deficit"])
plt.title("Meses con déficit por escenario")
plt.xlabel("Escenario")
plt.ylabel("Número de meses")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

# Gráfico 3: Déficit máximo por escenario
plt.figure(figsize=(11, 4))
plt.bar(tabla_escenarios["escenario"], tabla_escenarios["deficit_maximo"])
plt.title("Déficit máximo por escenario")
plt.xlabel("Escenario")
plt.ylabel("Déficit máximo (m³/s)")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

# Gráfico 4: Oferta y demanda para escenarios seleccionados
escenarios_seleccionados = [
    "Base",
    "Cambio climático (-15% oferta)",
    "Reúso de agua (+1.5 m³/s)",
]

for nombre in escenarios_seleccionados:
    df_temp = escenarios[nombre]
    plt.figure(figsize=(10, 4))
    plt.plot(df_temp["fecha"], df_temp["oferta_m3s"], marker="o", label="Oferta")
    plt.plot(df_temp["fecha"], df_temp["demanda_m3s"], marker="o", label="Demanda")
    plt.title(f"Oferta vs demanda — {nombre}")
    plt.xlabel("Fecha")
    plt.ylabel("Caudal equivalente (m³/s)")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Tests de validación — Fase 8

assert len(tabla_escenarios) == len(escenarios), "Debe existir información para todos los escenarios."

columnas_numericas_graficos = [
    "indice_resiliencia",
    "meses_deficit",
    "deficit_maximo",
]

for col in columnas_numericas_graficos:
    assert pd.api.types.is_numeric_dtype(tabla_escenarios[col]), f"La columna {col} debe ser numérica."

for nombre in escenarios_seleccionados:
    assert nombre in tabla_escenarios["escenario"].values, f"El escenario seleccionado {nombre} no existe en la tabla."

print("✅ Fase 8 validada: gráficos comparativos tienen datos consistentes.")

# Fase 9 — Interpretación técnica y toma de decisiones

## Objetivo

Convertir resultados numéricos en una decisión técnica de planificación hídrica.

## Preguntas guía

Responde en equipo o individualmente:

1. ¿Cuál escenario tiene mayor resiliencia?
2. ¿Cuál escenario tiene mayor déficit máximo?
3. ¿Qué medida de adaptación parece más efectiva?
4. ¿Qué medida parece insuficiente por sí sola?
5. ¿Qué combinación de medidas recomendarías para una ciudad como Lima?
6. ¿Qué indicador te parece más importante para tomar decisiones: confiabilidad, vulnerabilidad, recuperación o índice sintético? ¿Por qué?

## Conclusión técnica del estudiante

Escribe una conclusión de 5 a 8 líneas.  
Debe mencionar al menos:

- escenario más resiliente,
- principal escenario de riesgo,
- medida de adaptación recomendada,
- limitación del análisis,
- utilidad del índice para planificación.

> **Escribe aquí tu conclusión técnica:**  
>  
> ...

# Fase 10 — Validación final del notebook

## Objetivo

Ejecutar una validación general para verificar que el notebook se desarrolló correctamente de inicio a fin.

In [ ]:
# Tests finales del notebook

assert "df_base" in globals(), "df_base debe existir."
assert len(df_base) == 36, "df_base debe tener 36 filas."

escenarios_esperados = {
    "Base",
    "Cambio climático (-15% oferta)",
    "Crecimiento urbano (+20% demanda)",
    "Reducción de pérdidas (-10% demanda)",
    "Infraestructura natural (+8% oferta seca)",
    "Reúso de agua (+1.5 m³/s)",
    "Combinado CC+urbano+adaptación",
}

assert set(escenarios.keys()) == escenarios_esperados, "No se generaron todos los escenarios esperados."
assert not tabla_escenarios.isnull().any().any(), "tabla_escenarios no debe tener valores nulos."
assert tabla_escenarios["indice_resiliencia"].between(0, 1).all(), "Todos los índices deben estar entre 0 y 1."
assert any(df_esc["hay_deficit"].any() for df_esc in escenarios.values()), "Al menos un escenario debe presentar déficit."

indice_climatico = tabla_escenarios.loc[
    tabla_escenarios["escenario"] == "Cambio climático (-15% oferta)",
    "indice_resiliencia"
].iloc[0]

indices_adaptacion = tabla_escenarios.loc[
    tabla_escenarios["escenario"].isin([
        "Reducción de pérdidas (-10% demanda)",
        "Infraestructura natural (+8% oferta seca)",
        "Reúso de agua (+1.5 m³/s)",
        "Combinado CC+urbano+adaptación",
    ]),
    "indice_resiliencia"
]

assert (indices_adaptacion > indice_climatico).any(), (
    "Al menos un escenario de adaptación debe mejorar respecto al escenario de estrés climático."
)

print("Validación completada: el notebook se ejecutó correctamente")

# Conclusiones de la práctica

En este mini-proyecto se integraron conceptos de seguridad hídrica urbana con programación en Python.

Principales aprendizajes:

- La seguridad hídrica no depende solo de la oferta promedio, sino también de la relación dinámica entre oferta, demanda, déficit y recuperación.
- El crecimiento urbano y el cambio climático pueden aumentar la presión sobre el sistema.
- Las medidas de adaptación pueden evaluarse comparando indicadores comunes.
- Un índice sintético ayuda a comunicar resultados, pero siempre debe interpretarse junto con sus componentes.
- La planificación hídrica requiere combinar análisis técnico, criterios de resiliencia y juicio de ingeniería.

## Producto esperado del estudiante

Al finalizar, el estudiante debe entregar:

1. Notebook ejecutado.
2. Tabla comparativa de escenarios.
3. Tres gráficos principales.
4. Conclusión técnica argumentada.